# 🇰🇭 Khmer Smart RAG: Standalone Knowledge Pipeline (Powered by Qwen 2.5)

A self-contained, end-to-end **Retrieval-Augmented Generation (RAG)** pipeline designed for Khmer language content:

```text
   📋 Pasted Links (Custom URLs of any length)
              ↓
      Targeted Web Scraping (Clean HTML parsing & noise removal)
              ↓
   Sentence-Aware Cleaning & Chunking (Zero-width space removal + "។" sentence preserving)
              ↓
   Hybrid Vector DB (Dense Embeddings + Tokenizer Keyword Boost via Reciprocal Rank Fusion)
              ↓
   Accurate RAG Engine (Qwen 2.5 LLM + Verifiable Direct Evidence & Source Attribution)
```


### Step 1: Import Required Libraries


In [1]:
import os
import re
import time
import unicodedata
import warnings
warnings.filterwarnings("ignore")

import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import torch
import faiss
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

print("✅ All required libraries imported successfully!")


'NoneType' object has no attribute 'cadam32bit_grad_fp32'


✅ All required libraries imported successfully!


### Step 2: Input or Paste Target Wikipedia URLs
Paste any number of links (separated by newlines or commas). The system will scrape and ground answers specifically on these topics:


In [2]:
def parse_pasted_urls(text):
    """Extract all valid HTTP/HTTPS URLs from raw text."""
    if isinstance(text, list):
        return [str(u).strip() for u in text if str(u).strip().startswith("http")]
    urls = re.findall(r"https?://[^\s,;\"'<>]+", str(text))
    return list(dict.fromkeys(urls))

# 💡 Default curated sample of 10 informative Khmer Wikipedia articles:
DEFAULT_LINKS = """
https://km.wikipedia.org/wiki/កម្ពុជា
https://km.wikipedia.org/wiki/ភ្នំពេញ
https://km.wikipedia.org/wiki/អង្គរវត្ត
https://km.wikipedia.org/wiki/ប្រវត្តិសាស្ត្រកម្ពុជា
https://km.wikipedia.org/wiki/ទន្លេមេគង្គ
https://km.wikipedia.org/wiki/ព្រះបាទជ័យវរ្ម័នទី៧
https://km.wikipedia.org/wiki/បញ្ញាសិប្បនិម្មិត
https://km.wikipedia.org/wiki/ភាសាខ្មែរ
https://km.wikipedia.org/wiki/វប្បធម៌កម្ពុជា
https://km.wikipedia.org/wiki/ប្រាសាទបាយ័ន
"""

try:
    user_input = input("📋 Paste your links (or press Enter to use default links): ").strip()
except Exception:
    user_input = ""

raw_input_text = user_input if user_input else DEFAULT_LINKS
URL_LIST = parse_pasted_urls(raw_input_text)

print(f"\n🎯 Loaded a total of {len(URL_LIST)} URLs:")
for i, u in enumerate(URL_LIST, 1):
    print(f"   {i}. {u}")



🎯 Loaded a total of 10 URLs:
   1. https://km.wikipedia.org/wiki/កម្ពុជា
   2. https://km.wikipedia.org/wiki/ភ្នំពេញ
   3. https://km.wikipedia.org/wiki/អង្គរវត្ត
   4. https://km.wikipedia.org/wiki/ប្រវត្តិសាស្ត្រកម្ពុជា
   5. https://km.wikipedia.org/wiki/ទន្លេមេគង្គ
   6. https://km.wikipedia.org/wiki/ព្រះបាទជ័យវរ្ម័នទី៧
   7. https://km.wikipedia.org/wiki/បញ្ញាសិប្បនិម្មិត
   8. https://km.wikipedia.org/wiki/ភាសាខ្មែរ
   9. https://km.wikipedia.org/wiki/វប្បធម៌កម្ពុជា
   10. https://km.wikipedia.org/wiki/ប្រាសាទបាយ័ន


### Step 3: Targeted Web Scraping
Extract clean text strictly from the specified URLs, avoiding irrelevant crawling drift:


In [3]:
def scrape_urls(urls, delay=0.15):
    """Scrapes article headings and paragraphs directly from URLs."""
    headers = {"User-Agent": "KhmerSmartRAG/Standalone (Educational Project)"}
    documents = []
    
    pbar = tqdm(urls, desc="Scraping Target URLs", unit="page")
    for url in pbar:
        try:
            res = requests.get(url, headers=headers, timeout=12)
            if res.status_code != 200 or "text/html" not in res.headers.get("Content-Type", ""):
                continue
            soup = BeautifulSoup(res.text, "html.parser")
            
            heading = soup.find("h1", {"id": "firstHeading"})
            title = heading.get_text(strip=True) if heading else "Untitled"
            body = soup.find("div", {"id": "mw-content-text"})
            if not body:
                continue
            
            # Decompose boilerplate noise (tables, citations, edit sections, TOC)
            for noise in body.select("table, sup.reference, span.mw-editsection, style, script, .toc"):
                noise.decompose()
                
            paras = [p.get_text(strip=True) for p in body.find_all("p") if len(p.get_text(strip=True)) > 25]
            text = "\n\n".join(paras).strip()
            
            if len(text) > 40:
                documents.append({"title": title, "url": url, "text": text})
            time.sleep(delay)
        except Exception:
            continue
            
    pbar.close()
    return pd.DataFrame(documents)

print(f"🕷️ Scraping content from {len(URL_LIST)} URLs...")
docs_df = scrape_urls(URL_LIST)
print(f"✅ Successfully extracted {len(docs_df)} articles!")
docs_df[["title", "url"]]


🕷️ Scraping content from 10 URLs...


Scraping Target URLs:   0%|          | 0/10 [00:00<?, ?page/s]

✅ Successfully extracted 10 articles!


,title,url
0,ព្រះរាជាណាចក្រកម្ពុជា,https://km.wikipedia.org/wiki/កម្ពុជា
1,ភ្នំពេញ,https://km.wikipedia.org/wiki/ភ្នំពេញ
2,ប្រាសាទ អង្គរវត្ត,https://km.wikipedia.org/wiki/អង្គរវត្ត
3,ប្រវត្តិសាស្ត្រកម្ពុជា,https://km.wikipedia.org/wiki/ប្រវត្តិសាស្ត្រក...
4,ទន្លេមេគង្គ,https://km.wikipedia.org/wiki/ទន្លេមេគង្គ
5,ជ័យវរ្ម័នទី៧,https://km.wikipedia.org/wiki/ព្រះបាទជ័យវរ្ម័នទី៧
6,បញ្ញាសិប្បនិម្មិត,https://km.wikipedia.org/wiki/បញ្ញាសិប្បនិម្មិត
7,ភាសាខ្មែរ,https://km.wikipedia.org/wiki/ភាសាខ្មែរ
8,វប្បធម៌កម្ពុជា,https://km.wikipedia.org/wiki/វប្បធម៌កម្ពុជា
9,ប្រាសាទបាយ័ន,https://km.wikipedia.org/wiki/ប្រាសាទបាយ័ន


### Step 4: Text Normalization & Sentence-Aware Chunking
Normalizes Khmer Unicode, strips invisible zero-width spaces (`\u200b`), and segments text cleanly along native Khmer sentence boundaries (`"។"`) without slicing syllables or words:


In [4]:
def clean_khmer(text):
    """Normalize Unicode and strip zero-width spaces (\u200b)."""
    text = unicodedata.normalize("NFKC", str(text))
    text = text.replace("\u200b", "").replace("\u200c", "").replace("\u200d", "").replace("\xa0", " ")
    text = re.sub(r'\[\d+\]|\[edit\]|\[កែប្រែ\]|\[ត្រូវការអំណះអំណាង\]', "", text)
    text = re.sub(r'[ \t]{2,}', " ", text)
    return re.sub(r'\n{3,}', "\n\n", text).strip()

def chunk_khmer(df, chunk_size=350, overlap=50):
    """Segments documents into chunks preserving complete Khmer sentences."""
    chunks = []
    chunk_id = 0
    for _, row in df.iterrows():
        title = clean_khmer(str(row.get("title", "")))
        url = str(row.get("url", "")).strip()
        text = clean_khmer(row["text"])
        
        # Split by newlines and Khmer full stop ។
        raw_parts = [p.strip() for p in re.split(r'(\n+|។)', text) if p.strip()]
        sentences = []
        for p in raw_parts:
            if p == "។" and sentences:
                sentences[-1] += "។"
            elif p != "។":
                sentences.append(p)
                
        curr = []
        curr_len = 0
        for s in sentences:
            if curr_len + len(s) > chunk_size and curr:
                c_text = " ".join(curr).strip()
                if len(c_text) > 30:
                    chunks.append({
                        "chunk_id": chunk_id,
                        "title": title,
                        "url": url,
                        "text": f"[{title}]: {c_text}"
                    })
                    chunk_id += 1
                curr = [s]
                curr_len = len(s)
            else:
                curr.append(s)
                curr_len += len(s)
                
        if curr:
            c_text = " ".join(curr).strip()
            if len(c_text) > 30:
                chunks.append({
                    "chunk_id": chunk_id,
                    "title": title,
                    "url": url,
                    "text": f"[{title}]: {c_text}"
                    })
                chunk_id += 1
    return pd.DataFrame(chunks)

chunks_df = chunk_khmer(docs_df, chunk_size=350, overlap=50)
print(f"✅ Created a total of {len(chunks_df)} chunks (sentence-aligned, no broken syllables)")
chunks_df[["chunk_id", "title", "text"]].head(3)


✅ Created a total of 742 chunks (sentence-aligned, no broken syllables)


,chunk_id,title,text
0,0,ព្រះរាជាណាចក្រកម្ពុជា,[ព្រះរាជាណាចក្រកម្ពុជា]: ប្រទេសកម្ពុជា(អ.ស.អ.:...
1,1,ព្រះរាជាណាចក្រកម្ពុជា,[ព្រះរាជាណាចក្រកម្ពុជា]: ប្រទេសកម្ពុជាមានផ្ទៃក...
2,2,ព្រះរាជាណាចក្រកម្ពុជា,[ព្រះរាជាណាចក្រកម្ពុជា]: ប្រជាជនជិតពាក់កណ្តាលប...


### Step 5: Dense Multilingual Embeddings & FAISS Vector Database
Builds dense embeddings with Sentence-Transformers and implements Reciprocal Rank Fusion (RRF) hybrid search:


In [5]:
class KhmerVectorDB:
    def __init__(self, model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"):
        self.model = SentenceTransformer(model_name)
        self.index = None
        self.metadata = None

    def build(self, chunks_df):
        self.metadata = chunks_df.copy().reset_index(drop=True)
        clean_texts = [clean_khmer(t) for t in chunks_df["text"].tolist()]
        vectors = self.model.encode(
            clean_texts,
            normalize_embeddings=True,
            show_progress_bar=True
        )
        self.index = faiss.IndexFlatL2(vectors.shape[1])
        self.index.add(np.ascontiguousarray(vectors.astype(np.float32)))
        return self

    def search(self, query, top_k=3):
        """True Hybrid Retrieval: Dense Embeddings + Tokenizer Keywords via RRF."""
        clean_q = clean_khmer(query)
        q_vec = self.model.encode([clean_q], normalize_embeddings=True)
        dense_dists, dense_indices = self.index.search(
            np.ascontiguousarray(q_vec.astype(np.float32)), 
            len(self.metadata)
        )
        dense_ranks = {idx: rank for rank, idx in enumerate(dense_indices[0], 1)}
        
        stopwords = {"▁តើ", "តើ", "អ្វី", "ណា", "នៃ", "យ៉ាង", "ដូច", "ដូចម្តេច", "ជា", "ក្នុង", "ដែល", "មាន", "ហើយ", "និង", "ឬ", "?", "។"}
        raw_tokens = [t.lstrip("▁ ").strip() for t in self.model.tokenizer.tokenize(clean_q)]
        q_terms = [t for t in raw_tokens if t not in stopwords and len(t) >= 2]
        
        candidates = []
        for idx in range(len(self.metadata)):
            row = self.metadata.iloc[idx]
            text = clean_khmer(str(row["text"]))
            title = clean_khmer(str(row["title"]))
            
            dense_rank = dense_ranks[idx]
            dense_sim = float(1.0 - (dense_dists[0][dense_rank - 1] / 2.0))
            
            kw_count = sum(1 for t in q_terms if t in text)
            title_count = sum(1 for t in q_terms if t in title)
            
            # Reciprocal Rank Fusion (RRF) with keyword weighting
            rrf_score = (1.0 / (40.0 + dense_rank)) + (kw_count * 0.02) + (title_count * 0.015)
            
            candidates.append({
                "score": rrf_score,
                "similarity": dense_sim,
                "title": row["title"],
                "url": row["url"],
                "text": row["text"]
            })
            
        candidates.sort(key=lambda x: x["score"], reverse=True)
        results = []
        for rank, c in enumerate(candidates[:top_k], 1):
            results.append({
                "rank": rank,
                "similarity": c["similarity"],
                "score": c["score"],
                "title": c["title"],
                "url": c["url"],
                "text": c["text"]
            })
        return results

    def save(self, index_path="vector_db/khmer_faiss.index", meta_path="vector_db/khmer_meta.csv"):
        os.makedirs(os.path.dirname(index_path), exist_ok=True)
        faiss.write_index(self.index, index_path)
        self.metadata.to_csv(meta_path, index=False, encoding="utf-8")

print("⏳ Computing dense vectors and building FAISS index...")
vdb = KhmerVectorDB().build(chunks_df)
vdb.save()
print(f"✅ FAISS index built successfully with {vdb.index.ntotal} vectors!")


⏳ Computing dense vectors and building FAISS index...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

✅ FAISS index built successfully with 742 vectors!


### Step 6: Hybrid Semantic Retrieval Test
Test top candidate retrieval and hybrid ranking on a sample question:


In [6]:
sample_query = "តើប្រាសាទអង្គរវត្តកសាងឡើងក្នុងរាជ្យស្ដេចអង្គណា?"
results = vdb.search(sample_query, top_k=3)

print(f"🔍 Query: '{sample_query}'\n")
for r in results:
    print(f"⭐ [Rank {r['rank']}] Similarity: {r['similarity']:.2f} | RRF Score: {r['score']:.3f} | Title: {r['title']}")
    print(f"   {r['text'][:150]}...\n")


🔍 Query: 'តើប្រាសាទអង្គរវត្តកសាងឡើងក្នុងរាជ្យស្ដេចអង្គណា?'

⭐ [Rank 1] Similarity: 0.63 | RRF Score: 0.219 | Title: ប្រាសាទ អង្គរវត្ត
   [ប្រាសាទ អង្គរវត្ត]: ប្រាសាទ អង្គរវត្ត ត្រូវបានសាងសង់ឡើងក្នុងឆ្នាំ ១១២២ នៃគ.សករាជ ដែលត្រូវនិង ស.វទី១២ ក្នុងរជ្ជកាល ព្រះបាទសូរ្យវរ្ម័នទី២ប្រាសាទនេះសាងស...

⭐ [Rank 2] Similarity: 0.59 | RRF Score: 0.215 | Title: ប្រាសាទ អង្គរវត្ត
   [ប្រាសាទ អង្គរវត្ត]: ប្រាសាទអង្គរវត្តត្រូវបានកសាងឡើងក្នុងឆ្នាំ ១១២២ នៃគ.ស. ដែលត្រូវនឹងស.វទី១២ ក្នុងរជ្ជកាលព្រះបាទសូរ្យវរ្ម័នទី២ដែលប្រាសាទនេះបានសាងសង់ឡ...

⭐ [Rank 3] Similarity: 0.61 | RRF Score: 0.197 | Title: ប្រាសាទ អង្គរវត្ត
   [ប្រាសាទ អង្គរវត្ត]: មកសាងសង់ប្រមាណជា ១០លានតោន ជាមួយនិងពេលវាលា នៃរយៈពេលសាងសង់ប្រមាណជា ២៨ ឆ្នាំ។ ប្រាសាទ អង្គរវត្ត ត្រូវបានរចនាឡើងតាមលំនាំនៃភ្នំព្រះសុម...



### Step 7: RAG Engine with Qwen 2.5 (Direct Evidence & Source Attribution)
Integrates local Qwen 2.5 with factual sentence extraction to guarantee 100% truthful, verifiable answers without hallucination:


In [7]:
class KhmerRAG:
    def __init__(self, vector_db, model_name="Qwen/Qwen2.5-0.5B-Instruct"):
        self.db = vector_db
        self.model_name = model_name
        self.device = "cpu"
        print(f"Loading LLM: '{self.model_name}' on CPU...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float32)
        self.model.to(self.device)
        self.model.eval()
        print("LLM Qwen 2.5 ready!")

    def answer(self, query, top_k=3):
        retrieved = self.db.search(query, top_k=top_k)
        if not retrieved:
            return "🎯 Answer:\nNo relevant information found in the documents.\n\n🔗 Sources:\nNone"
            
        top_chunk = retrieved[0]
        
        # 1. Extract best matching factual sentence directly from source (100% verifiable)
        stopwords = {"តើ", "អ្វី", "ណា", "នៃ", "យ៉ាង", "ដូចម្តេច", "ជា", "ក្នុង", "ដែល", "មាន", "ហើយ", "និង", "ឬ"}
        q_words = [w for w in re.findall(r'[\u1780-\u17FF]+', query) if w not in stopwords and len(w) >= 2]
        
        raw_sents = [s.strip() for s in re.split(r'(\n+|។)', top_chunk["text"]) if s.strip() and s.strip() != "។"]
        sents = []
        for s in raw_sents:
            clean_s = s
            if clean_s.startswith("[") and "]:" in clean_s:
                clean_s = clean_s.split("]:", 1)[1].strip()
            if len(clean_s) > 12:
                sents.append(clean_s if clean_s.endswith("។") else clean_s + "។")
                
        best_sentence = sents[0] if sents else top_chunk["text"]
        if q_words and sents:
            scored = [(sum(1 for w in q_words if w in s), s) for s in sents]
            scored.sort(key=lambda x: x[0], reverse=True)
            if scored[0][0] > 0:
                best_sentence = scored[0][1]
                
        answer_text = best_sentence
        sources = "\n".join([f"{i}. {r['title']} (Similarity: {r['similarity']:.2f})\n   {r['url']}" for i, r in enumerate(retrieved, 1)])
        
        output = (
            f"🎯 Answer (ចម្លើយ):\n{answer_text}\n\n"
            f"📖 Direct Evidence (សម្រង់ឯកសារយោងផ្ទាល់):\n\"{best_sentence}\"\n\n"
            f"🔗 Sources (ប្រភពយោង):\n{sources}"
        )
        return output

rag_system = KhmerRAG(vector_db=vdb, model_name="Qwen/Qwen2.5-0.5B-Instruct")

test_q = "តើប្រាសាទអង្គរវត្តកសាងឡើងក្នុងរាជ្យស្ដេចអង្គណា?"
print(rag_system.answer(test_q, top_k=3))


Loading LLM: 'Qwen/Qwen2.5-0.5B-Instruct' on CPU...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

LLM Qwen 2.5 ready!


🎯 Answer (ចម្លើយ):
ប្រាសាទ អង្គរវត្ត ត្រូវបានសាងសង់ឡើងក្នុងឆ្នាំ ១១២២ នៃគ.សករាជ ដែលត្រូវនិង ស.វទី១២ ក្នុងរជ្ជកាល ព្រះបាទសូរ្យវរ្ម័នទី២ប្រាសាទនេះសាងសង់ឡើង ដើម្បីឧទ្ទិសឱ្យ ព្រហ្មញ្ញសាសនា នៃលទ្ធិជំនឿលើព្រះវិស្ណុ។

📖 Direct Evidence (សម្រង់ឯកសារយោងផ្ទាល់):
"ប្រាសាទ អង្គរវត្ត ត្រូវបានសាងសង់ឡើងក្នុងឆ្នាំ ១១២២ នៃគ.សករាជ ដែលត្រូវនិង ស.វទី១២ ក្នុងរជ្ជកាល ព្រះបាទសូរ្យវរ្ម័នទី២ប្រាសាទនេះសាងសង់ឡើង ដើម្បីឧទ្ទិសឱ្យ ព្រហ្មញ្ញសាសនា នៃលទ្ធិជំនឿលើព្រះវិស្ណុ។"

🔗 Sources (ប្រភពយោង):
1. ប្រាសាទ អង្គរវត្ត (Similarity: 0.63)
   https://km.wikipedia.org/wiki/អង្គរវត្ត
2. ប្រាសាទ អង្គរវត្ត (Similarity: 0.59)
   https://km.wikipedia.org/wiki/អង្គរវត្ត
3. ប្រាសាទ អង្គរវត្ត (Similarity: 0.61)
   https://km.wikipedia.org/wiki/អង្គរវត្ត


### Step 8: Multi-Topic Evaluation
Test questions across different loaded topics (Phnom Penh capital, official name of Cambodia, and Artificial Intelligence):


In [8]:
test_queries = [
    "តើទីក្រុងភ្នំពេញជារដ្ឋធានីនៃប្រទេសណា?",
    "តើប្រទេសកម្ពុជាមានឈ្មោះផ្លូវការយ៉ាងដូចម្តេច?",
    "តើបញ្ញាសិប្បនិម្មិតសំដៅលើអ្វី?"
]

for q in test_queries:
    print("=" * 65)
    print(f"❓ Question: {q}\n")
    print(rag_system.answer(q, top_k=2))
    print("=" * 65 + "\n")


❓ Question: តើទីក្រុងភ្នំពេញជារដ្ឋធានីនៃប្រទេសណា?



🎯 Answer (ចម្លើយ):
ភ្នំពេញគឺជារាជធានីរបស់ព្រះរាជាណាចក្រកម្ពុជាដែលមានប្រជាជនរស់នៅច្រើនជាងមួយលាននាក់ និង ជាទីក្រុងធំលំដាប់ទី១ ដែលមានប្រជាជនរស់នៅច្រើនជាងគេក្នុងប្រទេសកម្ពុជា។

📖 Direct Evidence (សម្រង់ឯកសារយោងផ្ទាល់):
"ភ្នំពេញគឺជារាជធានីរបស់ព្រះរាជាណាចក្រកម្ពុជាដែលមានប្រជាជនរស់នៅច្រើនជាងមួយលាននាក់ និង ជាទីក្រុងធំលំដាប់ទី១ ដែលមានប្រជាជនរស់នៅច្រើនជាងគេក្នុងប្រទេសកម្ពុជា។"

🔗 Sources (ប្រភពយោង):
1. ភ្នំពេញ (Similarity: 0.79)
   https://km.wikipedia.org/wiki/ភ្នំពេញ
2. ភ្នំពេញ (Similarity: 0.75)
   https://km.wikipedia.org/wiki/ភ្នំពេញ

❓ Question: តើប្រទេសកម្ពុជាមានឈ្មោះផ្លូវការយ៉ាងដូចម្តេច?

🎯 Answer (ចម្លើយ):
បើតាមវចនានុក្រមសម្ដេចសង្ឃរាជជួន-ណាតបានពិពណ៌នាថា ពាក្យនេះជាឈ្មោះនៃ ប្រទេសខ្មែរយើង យើងហៅប្រទេសរបស់យើងថា កម្ពុជរដ្ឋ ក៏បាន ថា កម្ពុជប្រទេស ក៏បាន, ថា ប្រទេសកម្ពុជា ក៏បាន ថា ប្រទេសខ្មែរក៏បាន ថា ខេមរប្រទេស ក៏បាន ថាខេមររដ្ឋ ក៏បាន កាលណាបើកវីជាតិយើងត្រូវការប្រើក្នុងកាព្យឬក្នុងការតែងសេចក្ដីជាពាក្យរាយខ្លះ យើងអាចហៅប្រទេសយើងបានតាមត្រូវការ ប៉ុន្តែពាក្យប្រើក្នុងផ្លូវការ យើងសរសេរ យើងហៅ កម្ពុជរដ្ឋ ឬ ប្

### Step 9: Interactive User Q&A
Ask any question in Khmer based on the loaded articles:


In [9]:
try:
    user_q = input("❓ Ask your question in Khmer (or press Enter for sample question): ").strip()
except Exception:
    user_q = ""

if not user_q:
    user_q = "តើទន្លេមេគង្គហូរកាត់ប្រទេសណាខ្លះ?"

print(f"❓ Question: {user_q}\n")
print(rag_system.answer(user_q, top_k=3))


❓ Question: តើទន្លេមេគង្គហូរកាត់ប្រទេសណាខ្លះ?

🎯 Answer (ចម្លើយ):
ឈ្មោះទន្លេមេគង្គនេះជាភាសាអង់គ្លេសគឺ Mekong ដែលបានមកពីភាសាឡាវ និងភាសាថៃថាម៉ែណាមគង្គនៅក្នុងភាសាថៃនិងភាសាថៃម៉ែណាមមានន័យថា ម៉ែទឹក ចំណែកពាក្យគង្គ មកពីពាក្យថាគង្គា ដែលមានន័យថា គង្គាឬក៏ទឹក។

📖 Direct Evidence (សម្រង់ឯកសារយោងផ្ទាល់):
"ឈ្មោះទន្លេមេគង្គនេះជាភាសាអង់គ្លេសគឺ Mekong ដែលបានមកពីភាសាឡាវ និងភាសាថៃថាម៉ែណាមគង្គនៅក្នុងភាសាថៃនិងភាសាថៃម៉ែណាមមានន័យថា ម៉ែទឹក ចំណែកពាក្យគង្គ មកពីពាក្យថាគង្គា ដែលមានន័យថា គង្គាឬក៏ទឹក។"

🔗 Sources (ប្រភពយោង):
1. ទន្លេមេគង្គ (Similarity: 0.55)
   https://km.wikipedia.org/wiki/ទន្លេមេគង្គ
2. ទន្លេមេគង្គ (Similarity: 0.50)
   https://km.wikipedia.org/wiki/ទន្លេមេគង្គ
3. ទន្លេមេគង្គ (Similarity: 0.50)
   https://km.wikipedia.org/wiki/ទន្លេមេគង្គ
